In [ ]:
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

In [ ]:
# Load the dataset from the CSV file
df = pd.read_csv("students.csv")

df.sample(5)

In [ ]:
df["result"].value_counts()

In [ ]:
# Select features and target
X = df[["study_hours", "attendance", "previous_score", "study_mode"]]
y = df["result"]

In [ ]:
# Define categorical and numeric features
categorical_features = ["study_mode"]
numeric_features = ["study_hours", "attendance", "previous_score"]

In [ ]:
# One-hot encode the categorical column, pass numbers through unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features),
    ]
)

In [ ]:
# Pipeline = preprocessing + model, so the API only needs ONE object
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42)),
])

In [ ]:
# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

In [ ]:
# Predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

In [ ]:
# Try one prediction (the API will do exactly this)
sample = pd.DataFrame([{
    "study_hours": 6.0, "attendance": 85.0, "previous_score": 70.0, "study_mode": "coaching"
}])
print(pipeline.predict(sample)[0])
print({cls: round(float(p), 3) for cls, p in zip(pipeline.classes_, pipeline.predict_proba(sample)[0])})

In [ ]:
# Save the trained pipeline using pickle
with open("model.pkl", "wb") as f:
    pickle.dump(pipeline, f)